# 1974 : la récession que le nominal ne voyait pas · *1974: the recession the nominal did not see*

Notebook compagnon du chapitre **10. PIB réel et PIB nominal : pourquoi corriger de l'inflation** — [lire l'article](https://nmlab.io/ressources/pib-reel-pib-nominal).
Companion notebook to chapter **10. Real GDP and Nominal GDP: Why Correct for Inflation** — [read the article](https://nmlab.io/en/ressources/real-gdp-nominal-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_growth() -> tuple[list[int], list[float], list[float]]:
    """Croissance annuelle du PIB américain nominal (GDPA) et réel (GDPCA), 1972-1976,
    calculée en direct depuis FRED (variation d'une année sur l'autre, en %).
    Annual growth of U.S. nominal (GDPA) and real (GDPCA) GDP, 1972-1976, live from FRED."""
    nominal = nm.load_fred("GDPA").pct_change() * 100
    real = nm.load_fred("GDPCA").pct_change() * 100
    years = list(range(1972, 1977))
    n = [round(float(nominal[nominal.index.year == y].iloc[0]), 1) for y in years]
    r = [round(float(real[real.index.year == y].iloc[0]), 1) for y in years]
    return years, n, r

years, nominal_growth, real_growth = load_growth()


import numpy as np
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="1974 : la récession que le nominal ne voyait pas",
        sub="Croissance du PIB américain — en dollars courants et en volumes",
        ylab="croissance annuelle, %",
        legend_nom="PIB nominal (dollars courants)",
        legend_real="PIB réel (volumes)",
        recession="1974-1975 : récession",
        note="En 1974 et 1975, le PIB nominal progresse de 8 à 9 % l'an pendant que les volumes reculent :\n"
             "toute la hausse — et davantage — n'était que des prix. Source : BEA via FRED (GDPA, GDPCA)."),
    "en": dict(
        title="1974: the recession the nominal did not see",
        sub="U.S. GDP growth — in current dollars and in volumes",
        ylab="annual growth, %",
        legend_nom="nominal GDP (current dollars)",
        legend_real="real GDP (volumes)",
        recession="1974-1975: recession",
        note="In 1974 and 1975 nominal GDP rises 8 to 9% a year while volumes shrink:\n"
             "all the increase — and more — was nothing but prices. Source: BEA via FRED (GDPA, GDPCA)."),
}

def _pct(value: float, lang: str) -> str:
    """Étiquette de pourcentage signée, localisée (« +9,8 % » / « +9.8% »)."""
    body = f"{value:+.1f}".replace(".", ",") if lang == "fr" else f"{value:+.1f}"
    return body + (" %" if lang == "fr" else "%")

def build_figure(years: list[int], nominal: list[float], real: list[float], lang: str) -> Figure:
    """Barres groupées par année : PIB nominal (bleu) et PIB réel (rose), bande de récession 1974-75."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1102)
    ax = nm.axes(fig, bottom=0.16)
    ax.grid(axis="x", visible=False)
    x = np.arange(len(years))
    width = 0.38
    ax.axvspan(1.5, 3.5, color=nm.COLORS["rose"], alpha=0.09, zorder=0)
    ax.bar(x - width / 2, nominal, width, color=nm.COLORS["blue"], label=text["legend_nom"], zorder=3)
    ax.bar(x + width / 2, real, width, color=nm.COLORS["rose"], label=text["legend_real"], zorder=3)
    ax.axhline(0, color=nm.COLORS["text"], lw=1.8, zorder=4)
    ax.set_ylim(-2.5, 16)
    ax.set_yticks(range(0, 16, 5))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(-0.6, len(years) - 0.4)
    ax.set_xticks(x)
    ax.set_xticklabels([str(y) for y in years], fontsize=24, color=nm.COLORS["muted"])
    ax.tick_params(axis="x", length=0)
    for pos, value in zip(x - width / 2, nominal):
        ax.annotate(_pct(value, lang), (pos, value), xytext=(0, 10), textcoords="offset points",
                    ha="center", va="bottom", fontsize=22, fontweight="bold", color=nm.COLORS["blue"])
    for pos, value in zip(x + width / 2, real):
        off, va = ((0, 10), "bottom") if value >= 0 else ((0, -12), "top")
        ax.annotate(_pct(value, lang), (pos, value), xytext=off, textcoords="offset points",
                    ha="center", va=va, fontsize=22, fontweight="bold", color=nm.COLORS["rose"])
    ax.text(2.5, 14.6, text["recession"], ha="center", va="center",
            fontsize=24, fontweight="bold", color=nm.COLORS["rose"])
    ax.legend(loc="upper left", frameon=False, fontsize=22, labelcolor=nm.COLORS["text"],
              handlelength=1.1, handleheight=1.1, borderaxespad=0.9)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(years, nominal_growth, real_growth, LANG)